# Features temporais por H3 — crimes e contexto socioeconômico

Agrega ocorrências por `H3` e `DATA`, completa os meses sem eventos, calcula features baseadas apenas no passado e preserva as features socioeconômicas.

## Entrada e estratégia de agregação

A entrada é `database_processed_h3_features_socio.csv`. As contagens de `CNT_CRIME`, `FURTO` e `ROUBO` são somadas por célula e mês. A grade completa usa todas as células H3 observadas e todos os meses entre a primeira e a última data; combinações sem ocorrência recebem zero. As features socioeconômicas vêm de setores censitários: cada combinação setor-H3 entra uma vez e os valores numéricos são resumidos pela média dos setores da célula. Indicadores `_missing` também são resumidos pela média, representando a fração de setores com valor imputado. `N_IPVS` e `CIDADE` usam a categoria modal.

In [1]:
import pandas as pd
import numpy as np
import h3
from pathlib import Path

input_path = Path('../datasets/database_processed_h3_features_socio.csv')
output_path = Path('../datasets/database_processed_h3_final.csv')


In [2]:
socio_cols = ['C_IPVS', 'pop_total', 'domicilios_total', 'media_moradores', 'domicilios_vagos', 'pop_masc', 'pop_fem', 'jovens_15_19', 'jovens_20_24', 'jovens_25_29', 'alfabetizados_15mais', 'nao_alfabetizados_15mais', 'pop_branca', 'pop_preta', 'pop_parda', 'dom_casa', 'dom_apartamento', 'dom_cortico', 'dom_degradado', 'dom_agua_encanada', 'dom_esgoto_rede', 'dom_lixo_coletado', 'pct_jovens_15_29', 'taxa_alfabetizacao', 'pct_negros']
missing_cols = ['C_IPVS_missing', 'pop_total_missing', 'domicilios_total_missing', 'media_moradores_missing', 'domicilios_vagos_missing', 'pop_masc_missing', 'pop_fem_missing', 'jovens_15_19_missing', 'jovens_20_24_missing', 'jovens_25_29_missing', 'alfabetizados_15mais_missing', 'nao_alfabetizados_15mais_missing', 'pop_branca_missing', 'pop_preta_missing', 'pop_parda_missing', 'dom_casa_missing', 'dom_apartamento_missing', 'dom_cortico_missing', 'dom_degradado_missing', 'dom_agua_encanada_missing', 'dom_esgoto_rede_missing', 'dom_lixo_coletado_missing', 'pct_jovens_15_29_missing', 'taxa_alfabetizacao_missing', 'pct_negros_missing']
crime_cols = ['CNT_CRIME', 'FURTO', 'ROUBO']


In [3]:
usecols = ['H3','DATA','CD_SETOR','CIDADE','N_IPVS','N_IPVS_missing'] + socio_cols + missing_cols + crime_cols
df_raw = pd.read_csv(input_path, usecols=usecols, low_memory=False)
df_raw['DATA'] = pd.to_datetime(df_raw['DATA'], format='%m-%Y')
df_raw = df_raw.dropna(subset=['H3','DATA'])
df_raw['H3'] = df_raw['H3'].astype(str)


## Agregar ocorrências e completar a grade H3 × mês

In [4]:
crime = df_raw.groupby(['H3','DATA'], observed=True)[crime_cols].sum()
cells = sorted(df_raw['H3'].unique())
months = pd.date_range(df_raw['DATA'].min(), df_raw['DATA'].max(), freq='MS')
grid = pd.MultiIndex.from_product([cells, months], names=['H3','DATA'])
df_processed = crime.reindex(grid, fill_value=0).reset_index()


## Resumir as features socioeconômicas por H3

Cada setor censitário contribui uma vez em cada H3. Valores numéricos são promediados; campos categóricos usam a moda.

In [5]:
sector_cols = ['CD_SETOR','H3'] + socio_cols + missing_cols + ['N_IPVS','N_IPVS_missing','CIDADE']
sectors = df_raw[sector_cols].drop_duplicates(['CD_SETOR','H3'])
num_cols = socio_cols + missing_cols + ['N_IPVS_missing']
socio = sectors.groupby('H3', observed=True)[num_cols].mean()
for col in ['N_IPVS','CIDADE']:
    socio[col] = sectors.groupby('H3', observed=True)[col].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
socio = socio.reset_index()
df_processed = (df_processed.merge(socio, on='H3', how='left')
                .sort_values(['H3','DATA']).reset_index(drop=True))


## Criar features históricas e de vizinhança

Lags e médias móveis são deslocados para usar somente meses anteriores. As features de vizinhança também usam os três meses anteriores.

In [6]:
targets = {'crime':'CNT_CRIME', 'furto':'FURTO', 'roubo':'ROUBO'}
grouped = df_processed.groupby('H3', sort=False)
for prefix, col in targets.items():
    lag1 = grouped[col].shift(1)
    roll3 = lag1.groupby(df_processed['H3'], sort=False).transform(lambda s: s.rolling(3, min_periods=3).mean())
    roll12 = lag1.groupby(df_processed['H3'], sort=False).transform(lambda s: s.rolling(12, min_periods=12).mean())
    lag12 = grouped[col].shift(12)
    occurred = grouped[col].shift(1).gt(0).astype(float)
    pct12 = occurred.groupby(df_processed['H3'], sort=False).transform(lambda s: s.rolling(12, min_periods=12).mean())
    df_processed[f'{prefix}_media_12m_log'] = np.log1p(roll12)
    df_processed[f'{prefix}_razao_3_12'] = (roll3 + 0.5) / (roll12 + 0.5)
    df_processed[f'{prefix}_sazonal_12'] = (lag12 + 0.5) / (roll12 + 0.5)
    df_processed[f'{prefix}_pct_meses_12m'] = pct12

cell_set = set(cells)
neighbors = {cell: sorted((set(h3.grid_disk(cell, 1)) - {cell}) & cell_set) for cell in cells}
for prefix, col in targets.items():
    wide = df_processed.pivot(index='DATA', columns='H3', values=col).reindex(index=months, columns=cells, fill_value=0)
    neighbor_monthly = pd.DataFrame(index=months, columns=cells, dtype=float)
    for cell in cells:
        adjacent = neighbors[cell]
        neighbor_monthly[cell] = wide[adjacent].mean(axis=1) if adjacent else 0.0
    neighbor_prior3 = np.log1p(neighbor_monthly.shift(1).rolling(3, min_periods=3).mean())
    neighbor_long = neighbor_prior3.stack().rename(f'{prefix}_viz_media_3m_log').rename_axis(index=['DATA','H3']).reset_index()
    df_processed = df_processed.merge(neighbor_long, on=['DATA','H3'], how='left')
    df_processed[f'{prefix}_viz_rel'] = df_processed[f'{prefix}_viz_media_3m_log'] - df_processed[f'{prefix}_media_12m_log']


## Salvar e conferir

In [7]:
df_processed.to_csv(output_path, index=False)
print(f'Base salva em: {output_path.resolve()}')
print('Linhas:', len(df_processed))
print('Células H3:', df_processed['H3'].nunique())
print('Meses:', df_processed['DATA'].nunique())
print('Combinações com zero crime:', int(df_processed['CNT_CRIME'].eq(0).sum()))
print('Features socioeconômicas numéricas ausentes:', int(df_processed[socio_cols].isna().sum().sum()))


Base salva em: C:\Users\Usuario\Documents\USP\8_SEMESTRE\Resolução de Problemas II\sp-crime-spatiotemporal-prediction\datasets\database_processed_h3_final.csv
Linhas: 68040
Células H3: 1260
Meses: 54
Combinações com zero crime: 29211
Features socioeconômicas numéricas ausentes: 0
